# Sales Data EDA

A beginner-friendly exploratory analysis of reproducible synthetic sales data. Each row represents one order. The data is generated locally and does not represent a real business.

## 1. Imports and setup

Run the notebook cells from top to bottom. The fixed random seed makes the sample data repeatable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Create synthetic sales data

This generator uses the same fields, categories, regions, date range, and seed as the Streamlit dashboard.

In [ ]:
rng = np.random.default_rng(42)
row_count = 600
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
categories = ["Electronics", "Home", "Clothing", "Beauty"]
regions = ["North", "South", "East", "West"]
category_prices = {
    "Electronics": (180, 900),
    "Home": (35, 300),
    "Clothing": (20, 180),
    "Beauty": (10, 100),
}

sales = pd.DataFrame(
    {
        "Date": rng.choice(dates, row_count),
        "Category": rng.choice(categories, row_count),
        "Region": rng.choice(regions, row_count),
        "Units": rng.integers(1, 6, row_count),
    }
)
sales["Unit Price"] = sales["Category"].map(
    lambda category: rng.uniform(*category_prices[category])
)
sales["Revenue"] = (sales["Units"] * sales["Unit Price"]).round(2)
sales = sales.sort_values("Date").reset_index(drop=True)

sales.head()

## 3. First look at the data

Check the number of rows and columns, data types, and a sample of the records.

In [ ]:
print(f"Rows: {sales.shape[0]:,} | Columns: {sales.shape[1]}")
display(sales.sample(5, random_state=42))
sales.info()

## 4. Data quality checks

The missing-value and exact-duplicate counts are useful checks. Exact duplicate rows can occur by chance in synthetic data; they are not automatically errors because the dataset has no unique order ID.

In [ ]:
print("Missing values by column:")
display(sales.isna().sum().to_frame("Missing values"))
print(f"Exact duplicate rows: {sales.duplicated().sum()}")
print(f"Date range: {sales['Date'].min():%Y-%m-%d} to {sales['Date'].max():%Y-%m-%d}")

## 5. Descriptive statistics

Review the distribution of units, unit prices, and revenue, then check how many orders belong to each category and region.

In [ ]:
display(sales[["Units", "Unit Price", "Revenue"]].describe().T)
print("Orders by category:")
display(sales["Category"].value_counts().rename("Orders").to_frame())
print("Orders by region:")
display(sales["Region"].value_counts().rename("Orders").to_frame())

## 6. Overall sales performance

Summarize total revenue, order count, average order value, and units sold.

In [ ]:
summary = pd.Series(
    {
        "Total revenue": sales["Revenue"].sum(),
        "Orders": len(sales),
        "Average order value": sales["Revenue"].mean(),
        "Units sold": sales["Units"].sum(),
    }
)
display(summary.to_frame("Value"))

## 7. Revenue over time

Group orders by month to see how revenue is distributed across the year.

In [ ]:
monthly_revenue = (
    sales.assign(Month=sales["Date"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month")["Revenue"]
    .sum()
)
display(monthly_revenue.rename("Revenue").to_frame())

ax = monthly_revenue.plot(kind="line", marker="o", figsize=(10, 4), color="teal")
ax.set(title="Monthly Revenue", xlabel="Month", ylabel="Revenue ($)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Category and region comparisons

Compare total revenue across categories and regions. These totals show which groups contribute the most revenue in this synthetic sample.

In [ ]:
category_revenue = sales.groupby("Category")["Revenue"].sum().sort_values(ascending=False)
region_revenue = sales.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
display(category_revenue.rename("Revenue").to_frame())
display(region_revenue.rename("Revenue").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
category_revenue.plot(kind="bar", ax=axes[0], color="cornflowerblue")
axes[0].set(title="Revenue by Category", xlabel="Category", ylabel="Revenue ($)")
region_revenue.plot(kind="bar", ax=axes[1], color="darkseagreen")
axes[1].set(title="Revenue by Region", xlabel="Region", ylabel="Revenue ($)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Takeaways

- Use the monthly chart to spot higher- and lower-revenue months.
- Compare category and region totals to see where synthetic revenue is concentrated.
- Because the data is randomly generated, patterns are examples for practice and should not be treated as real-world findings.